Le projet ne repose pas sur un modèle de classification classique mais sur un système de recommandation collaborative. L’objectif n’est pas de prédire une classe, mais d’identifier des similarités entre livres à partir des interactions utilisateurs-livres afin de proposer des recommandations personnalisées.

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from dotenv import load_dotenv
# Ajouter la racine du projet au PYTHONPATH
PROJECT_ROOT = Path().cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)

# Charger le .env
load_dotenv(PROJECT_ROOT / "config" / ".env")

print("ENV loaded from:", PROJECT_ROOT / "config" / ".env")

from src.utils.s3_io import read_parquet_from_s3

Project root added: c:\Users\Administrateur\Desktop\BiblioTech
ENV loaded from: c:\Users\Administrateur\Desktop\BiblioTech\config\.env


In [2]:
# Chargement des données 

ratings = read_parquet_from_s3("silver", "ratings_joinable.parquet")

print(ratings.shape)
ratings.head()

(382769, 3)


,user_id,isbn,rating
0,276726,0155061224,5
1,276729,052165615X,3
2,276729,0521795028,6
3,276744,038550120X,7
4,276747,0060517794,9


### Dataset de départ

Le dataset contient les interactions utilisateur-livre (ratings explicites).

Chaque ligne correspond à :
- un utilisateur
- un livre
- une note entre 1 et 10

Ce format est la base d’un système de recommandation collaborative.

In [3]:
# Statistiques de base 

n_users = ratings["user_id"].nunique()
n_books = ratings["isbn"].nunique()
n_ratings = len(ratings)

print("Nombre d'utilisateurs :", n_users)
print("Nombre de livres :", n_books)
print("Nombre de ratings :", n_ratings)

Nombre d'utilisateurs : 67934
Nombre de livres : 149222
Nombre de ratings : 382769


### Vue globale

On mesure ici le nombre d’utilisateurs, de livres et d’interactions disponibles.

Ces trois valeurs permettent d’évaluer la taille théorique de la matrice utilisateur-livre.

## PARTIE 1 — Calcul de la sparsité sans créer la matrice dense

In [4]:
# Création de la matrice utilisateur-livre 

user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="isbn",
    values="rating"
)

user_item_matrix.head()

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_7968\1032481295.py:3: PerformanceWarning: The following operation may generate 10137247348 cells in the resulting pandas object.
  user_item_matrix = ratings.pivot_table(


MemoryError: Unable to allocate 75.5 GiB for an array with shape (67934, 149222) and data type float64

### Limite de la matrice dense

La tentative de création d’une matrice utilisateur-livre avec `pivot_table` provoque une erreur mémoire.

Une matrice utilisateur-livre complète contiendrait une note pour chaque couple utilisateur/livre.

Or, dans la réalité :
- chaque utilisateur ne note qu’une petite partie des livres
- chaque livre n’est noté que par une petite partie des utilisateurs

Même si seules quelques centaines de milliers de notes existent réellement, Pandas essaierait de créer toutes les cellules, y compris les valeurs manquantes.

Cela provoque une erreur mémoire.

Cela montre que la matrice utilisateur-livre est extrêmement creuse (*sparse*).
La sparsité mesure la proportion de valeurs manquantes dans la matrice utilisateur-livre.

### Solution retenue : création d'une matrice sparse

Au lieu de créer une matrice dense avec Pandas, on utilise une matrice creuse (`csr_matrix`) de SciPy.

Cette structure ne stocke que les interactions réellement observées, ce qui permet de représenter efficacement les données pour les modèles de recommandation.

In [5]:
from scipy.sparse import csr_matrix

# Encodage des identifiants utilisateurs et livres
user_ids = ratings["user_id"].astype("category")
book_ids = ratings["isbn"].astype("category")

user_index = user_ids.cat.codes
book_index = book_ids.cat.codes

# Création de la matrice sparse
user_item_sparse = csr_matrix(
    (ratings["rating"], (user_index, book_index)),
    shape=(user_ids.cat.categories.size, book_ids.cat.categories.size)
)

user_item_sparse.shape

(67934, 149222)

### Matrice utilisateur-livre sparse

La matrice sparse représente :

- les lignes : utilisateurs
- les colonnes : livres
- les valeurs : ratings

Contrairement à une matrice classique, elle ne stocke que les valeurs existantes.

C’est le format adapté aux systèmes de recommandation.

## PARTIE 2 — SPARSITÉ

In [6]:
# Mesure de la sparsité 

num_users, num_books = user_item_sparse.shape
num_actual = user_item_sparse.nnz # nimbre de valuers stockées
num_possible = num_users * num_books

sparsity = 1 - (num_actual / num_possible)

print(f"Users: {num_users}")
print(f"Books: {num_books}")
print(f"Ratings réels: {num_actual}")
print(f"Cellules possibles: {num_possible}")
print(f"Sparsité: {sparsity:.6%}")

Users: 67934
Books: 149222
Ratings réels: 382769
Cellules possibles: 10137247348
Sparsité: 99.996224%


### Sparsité de la matrice

La sparsité mesure le pourcentage de valeurs manquantes dans la matrice.

Résultat :
- très proche de 1 (≈ 99% vide)

### Interprétation

- la majorité des utilisateurs n’ont pas noté la majorité des livres
- les données sont extrêmement creuses

C’est une caractéristique typique des systèmes de recommandation.

## PARTIE 3 — FILTRAGE

### Filtrer les utilisateurs actifs

In [7]:
MIN_USER_RATINGS = 5

user_counts = ratings["user_id"].value_counts()
active_users = user_counts[user_counts >= MIN_USER_RATINGS].index

ratings_filtered = ratings[ratings["user_id"].isin(active_users)].copy()

print("Avant filtrage utilisateurs :", ratings.shape)
print("Après filtrage utilisateurs :", ratings_filtered.shape)
print("Utilisateurs conservés :", ratings_filtered["user_id"].nunique())

Avant filtrage utilisateurs : (382769, 3)
Après filtrage utilisateurs : (301321, 3)
Utilisateurs conservés : 12760


#### Filtrage des utilisateurs

On conserve uniquement les utilisateurs ayant au moins 5 ratings.

Objectif :
- réduire le bruit
- limiter le problème de cold start
- garder les utilisateurs pour lesquels on peut identifier un minimum de préférences

In [8]:
# Nombre de ratings par livre après filtrage utilisateurs
book_counts = ratings_filtered["isbn"].value_counts()

book_counts.describe()

count    130853.000000
mean          2.302744
std           5.671377
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         430.000000
Name: count, dtype: float64

### Popularité des livres

Après avoir filtré les utilisateurs peu actifs, on analyse le nombre de ratings par livre.

Beaucoup de livres restent très peu notés, ce qui rend leur évaluation peu fiable.

### Filtrer les livres suffisamment notés

In [9]:
MIN_BOOK_RATINGS = 5

popular_books = book_counts[book_counts >= MIN_BOOK_RATINGS].index

ratings_filtered = ratings_filtered[ratings_filtered["isbn"].isin(popular_books)].copy()

print("Dataset initial :", ratings.shape)
print("Dataset filtré :", ratings_filtered.shape)
print("Utilisateurs conservés :", ratings_filtered["user_id"].nunique())
print("Livres conservés :", ratings_filtered["isbn"].nunique())

Dataset initial : (382769, 3)
Dataset filtré : (135024, 3)
Utilisateurs conservés : 12088
Livres conservés : 10641


### Filtrage des livres

On conserve uniquement les livres ayant au moins 5 ratings.

Objectif :
- éviter les moyennes peu fiables
- réduire la sparsité
- préparer un dataset plus robuste pour le machine learning

### Sparsité après filtrage

In [10]:
n_users_filtered = ratings_filtered["user_id"].nunique()
n_books_filtered = ratings_filtered["isbn"].nunique()
n_ratings_filtered = len(ratings_filtered)

n_possible_filtered = n_users_filtered * n_books_filtered
sparsity_filtered = 1 - (n_ratings_filtered / n_possible_filtered)

print("Utilisateurs après filtrage :", n_users_filtered)
print("Livres après filtrage :", n_books_filtered)
print("Ratings après filtrage :", n_ratings_filtered)
print("Cellules possibles après filtrage :", n_possible_filtered)
print(f"Sparsité après filtrage : {sparsity_filtered:.6%}")

Utilisateurs après filtrage : 12088
Livres après filtrage : 10641
Ratings après filtrage : 135024
Cellules possibles après filtrage : 128628408
Sparsité après filtrage : 99.895028%


### Sparsité après filtrage

Le filtrage réduit le nombre d’utilisateurs et de livres, mais conserve les interactions les plus informatives.

Cela permet d’obtenir un dataset plus dense et plus adapté à l’apprentissage.

Il y a un compromis :
- on perd une partie des données
- mais on améliore leur qualité pour le modèle

### Création de la matrice sparse filtrée

In [11]:
user_ids_f = ratings_filtered["user_id"].astype("category")
book_ids_f = ratings_filtered["isbn"].astype("category")

user_index_f = user_ids_f.cat.codes
book_index_f = book_ids_f.cat.codes

user_item_sparse_filtered = csr_matrix(
    (ratings_filtered["rating"], (user_index_f, book_index_f)),
    shape=(user_ids_f.cat.categories.size, book_ids_f.cat.categories.size)
)

print("Shape matrice filtrée :", user_item_sparse_filtered.shape)
print("Valeurs stockées :", user_item_sparse_filtered.nnz)

Shape matrice filtrée : (12088, 10641)
Valeurs stockées : 135024


### Matrice sparse filtrée

Cette matrice est la version préparée pour les futurs modèles de recommandation.

Elle contient uniquement :
- des utilisateurs suffisamment actifs
- des livres suffisamment notés

Elle sera plus exploitable pour les algorithmes de recommandation collaborative.

### Matrice sparse filtrée

Cette matrice est la version préparée pour les futurs modèles de recommandation.

Elle contient uniquement :
- des utilisateurs suffisamment actifs
- des livres suffisamment notés

Elle sera plus exploitable pour les algorithmes de recommandation collaborative.

In [12]:
user_mapping = pd.DataFrame({
    "user_id": user_ids_f.cat.categories,
    "user_index": range(len(user_ids_f.cat.categories))
})

book_mapping = pd.DataFrame({
    "isbn": book_ids_f.cat.categories,
    "book_index": range(len(book_ids_f.cat.categories))
})

user_mapping.head(), book_mapping.head()

(   user_id  user_index
 0        8           0
 1       99           1
 2      114           2
 3      242           3
 4      243           4,
          isbn  book_index
 0  0002005018           0
 1  0002251760           1
 2  0006480764           2
 3  000648302X           3
 4  000649840X           4)

### Mappings utilisateurs et livres

Les modèles de recommandation utilisent des indices numériques.

Les mappings permettent de relier :
- les identifiants réels (`user_id`, `isbn`)
- aux indices internes utilisés dans la matrice sparse

Ces mappings seront nécessaires pour interpréter les recommandations.

### Sauvegarde optionnelle des datasets préparés

In [ ]:
# localement 
ratings_filtered.to_parquet("../../data/gold/ratings_pre_ml.parquet", index=False)
user_mapping.to_parquet("../../data/gold/user_mapping.parquet", index=False)
book_mapping.to_parquet("../../data/gold/book_mapping.parquet", index=False)

from src.utils.s3_io import write_parquet_to_s3, list_objects

write_parquet_to_s3(ratings_filtered,"gold","ratings_pre_ml.parquet")

write_parquet_to_s3(user_mapping, "gold", "user_mapping.parquet")
write_parquet_to_s3(book_mapping, "gold", "book_mapping.parquet")

list_objects("gold")

['book_popularity.parquet', 'ratings_pre_ml.parquet']

### Conclusion Pre-ML

Cette étape a permis de transformer les données de ratings en structure exploitable pour le machine learning.

Les points clés sont :

- la matrice utilisateur-livre est extrêmement sparse
- une matrice dense Pandas n’est pas adaptée
- une matrice sparse SciPy est nécessaire
- le filtrage améliore la qualité du dataset
- les mappings utilisateurs/livres permettent de préparer les futurs modèles

Cette préparation constitue la base du futur système de recommandation collaborative.